### 1. AutoGen- open-source programming framework developed by Microsoft Research
### 2. Using Ollam with Autogen

In [48]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [3]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [18]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2:1b",
host="http://172.25.16.1:11434")


In [61]:
from autogen_agentchat.messages import TextMessage
message = TextMessage(content="I'd like to go to Bercelona", source="user")
message

TextMessage(id='f8ff6aaf-6492-4342-aa15-7431e5bf9100', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 3, 24, 7, 36, 0, 712456, tzinfo=datetime.timezone.utc), content="I'd like to go to Bercelona", type='TextMessage')

In [62]:
from autogen_agentchat.agents import AssistantAgent

agent = AssistantAgent(
    name="airline_agent",
    model_client=ollamamodel_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers.",
    model_client_stream=True
)

In [63]:
from autogen_core import CancellationToken

response = await agent.on_messages([message], cancellation_token=CancellationToken())
response.chat_message.content

'Bercelona (that\'s Barcelona for you non-Spanish speakers) is a great destination. Are you looking forward to the beaches, tapas, or maybe trying to pronounce "siesta" without getting it wrong?'

In [22]:
import os
import sqlite3

# Delete existing database file if it exists
if os.path.exists("tickets.db"):
    os.remove("tickets.db")

# Create the database and the table
conn = sqlite3.connect("tickets.db")
c = conn.cursor()
c.execute("CREATE TABLE cities (city_name TEXT PRIMARY KEY, round_trip_price REAL)")
conn.commit()
conn.close()

In [53]:
# Populate our database
def save_city_price(city_name, round_trip_price):
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("REPLACE INTO cities (city_name, round_trip_price) VALUES (?, ?)", (city_name.lower(), round_trip_price))
    conn.commit()
    conn.close()

# Some cities!
save_city_price("London", 299)
save_city_price("Paris", 399)
save_city_price("Rome", 499)
save_city_price("Madrid", 550)
save_city_price("Barcelona", 586)
save_city_price("Barcelona", 580)
save_city_price("Berlin", 525)

In [30]:
# Method to get price for a city
def get_city_price(city_name: str) -> float | None:
    """ Get the roundtrip ticket price to travel to the city """
    conn = sqlite3.connect("tickets.db")
    c = conn.cursor()
    c.execute("SELECT round_trip_price FROM cities WHERE city_name = ?", (city_name.lower(),))
    result = c.fetchone()
    conn.close()
    return result[0] if result else None

In [31]:
get_city_price("London")

299.0

In [66]:
from autogen_agentchat.agents import AssistantAgent

smart_agent = AssistantAgent(
    name="smart_airline_agent",
    model_client=model_client,
    system_message="You are a helpful assistant for an airline. You give short, humorous answers, including the price of a roundtrip ticket.",
    model_client_stream=True,
    tools=[get_city_price],
    reflect_on_tool_use=True
)

In [57]:
response = await smart_agent.on_messages([message], cancellation_token=CancellationToken())
for inner_message in response.inner_messages:
    print(inner_message.content)
response.chat_message.content

print(response.chat_message.content)

[FunctionCall(id='call_ZzpP5EGCaw3O9u3VyJMSIaN7', arguments='{"city_name":"London"}', name='get_city_price')]
[FunctionExecutionResult(content='299.0', name='get_city_price', call_id='call_ZzpP5EGCaw3O9u3VyJMSIaN7', is_error=False)]
Sure! A roundtrip ticket to London will cost you $299. Just think of it as investing in a great story—“I survived the plane food!” 🛫🥪


In [67]:
from autogen_core import CancellationToken
from autogen_agentchat.messages import TextMessage

message2 = TextMessage(content="How much is a flight to Bercelona?.", source="user")
# 1. Await the full response
esponse = await smart_agent.on_messages(
    [message2],
    cancellation_token=CancellationToken()
)
print(response.chat_message.content)


Bercelona (that's Barcelona for you non-Spanish speakers) is a great destination. Are you looking forward to the beaches, tapas, or maybe trying to pronounce "siesta" without getting it wrong?
